# PQID Scientific Data Quantitative Figure Notebook

This notebook generates the quantitative plot figures proposed for the Scientific Data submission. The Mermaid diagrams in `figures/` describe workflow logic; this notebook covers statistical and release-composition evidence.

Generated outputs are written to `PQID/submissions/scientific_data/figures/` as SVG and PNG files.

In [ ]:
from __future__ import annotations

import ast
import json
import os
from collections import Counter
from pathlib import Path

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "PQID").exists() and (candidate / "PQID" / "data").exists():
            return candidate
        if candidate.name == "PQID" and (candidate / "data").exists():
            return candidate.parent
    raise RuntimeError("Could not locate repository root containing PQID/data.")


ROOT = find_repo_root()
PQID_ROOT = ROOT / "PQID"
PROCESSED_DIR = PQID_ROOT / "data" / "processed"
_figure_dir_override = os.environ.get("PQID_SCI_DATA_FIGURE_DIR")
if _figure_dir_override:
    _figure_dir_path = Path(_figure_dir_override)
    FIGURE_DIR = (
        _figure_dir_path
        if _figure_dir_path.is_absolute()
        else PQID_ROOT / "submissions" / "scientific_data" / _figure_dir_path
    )
else:
    FIGURE_DIR = PQID_ROOT / "submissions" / "scientific_data" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_OUTPUT_STEMS = [
    "fig5_readiness_statistics",
    "fig6_semantic_paraphrase_quality",
    "fig7_release_composition",
    "suppfig_s4_acquisition_pareto_diminishing_returns",
    "suppfig_s5_linguistic_distribution",
    "suppfig_s6_license_behavior_panel",
]

_font_override = os.environ.get("PQID_SCI_DATA_FIGURE_FONT")
if _font_override:
    FONT_FAMILY = "sans-serif"
    FONT_STACK = [_font_override, "Calibri", "Arial", "Liberation Sans", "DejaVu Sans"]
else:
    FONT_FAMILY = "serif"
    FONT_STACK = ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"]
AVAILABLE_FONTS = {font.name for font in font_manager.fontManager.ttflist}
MANUSCRIPT_FONT = next((font for font in FONT_STACK if font in AVAILABLE_FONTS), "DejaVu Sans" if _font_override else "DejaVu Serif")


def read_jsonl(path: Path, max_rows: int | None = None):
    with path.open(encoding="utf-8") as handle:
        for idx, line in enumerate(handle, start=1):
            if line.strip():
                yield json.loads(line)
            if max_rows is not None and idx >= max_rows:
                break


def parse_list_like(value) -> list[str]:
    if value is None or value == "":
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(text)
            except Exception:
                continue
            if isinstance(parsed, list):
                return [str(item) for item in parsed]
        return [text]
    return [str(value)]


def as_int(value):
    try:
        if value is None or value == "":
            return np.nan
        return int(value)
    except Exception:
        return np.nan


def poisson_binomial_pmf(probabilities: np.ndarray) -> np.ndarray:
    pmf = np.array([1.0])
    for p in probabilities:
        pmf = np.convolve(pmf, np.array([1.0 - p, p]))
    return pmf


def thousands(x, _pos=None):
    return f"{int(x):,}"


def save_and_display_figure(fig, stem: str):
    svg = FIGURE_DIR / f"{stem}.svg"
    png = FIGURE_DIR / f"{stem}.png"
    fig.savefig(svg, bbox_inches="tight")
    fig.savefig(png, dpi=450, bbox_inches="tight")
    display(Markdown(
        f"**Saved `{stem}`**  \n"
        f"- SVG: `{svg}`  \n"
        f"- PNG: `{png}`"
    ))
    display(fig)
    plt.close(fig)
    return {"svg": svg, "png": png}


COLORS = {
    "blue": "#3b6ea8",
    "green": "#3f8f5f",
    "amber": "#c9852b",
    "red": "#b94a48",
    "slate": "#4b5563",
    "light": "#eef2f7",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.labelcolor": "#111111",
    "xtick.color": "#111111",
    "ytick.color": "#111111",
    "font.family": FONT_FAMILY,
    "font.serif": [MANUSCRIPT_FONT, "Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "font.sans-serif": [MANUSCRIPT_FONT, "Calibri", "Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "mathtext.fontset": "dejavusans" if _font_override else "stix",
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

display(Markdown(
    "**Figure output configuration**  \n"
    f"- Output directory: `{FIGURE_DIR}`  \n"
    f"- Plot font: `{MANUSCRIPT_FONT}`  \n"
    "- Planned files: " + ", ".join(f"`{stem}.svg` / `{stem}.png`" for stem in FIGURE_OUTPUT_STEMS)
))


## Figure 5: Benchmark-Readiness Statistics

This multi-panel figure summarizes n/7 and n/8 readiness scores, compares the observed n/8 distribution against a Poisson-binomial null, and plots the phi/correlation matrix of readiness checks.

In [ ]:
MASTER_FILE = PROCESSED_DIR / "pqid_2026_master_corpus.jsonl"

N7_CHECKS = [
    "validated_execution",
    "high_extraction_confidence",
    "no_demo_scaffolding",
    "no_cleanup_candidate",
    "minimum_code_lines",
    "minimum_gate_count",
    "trusted_retrieval_strategy",
]
N8_CHECKS = [*N7_CHECKS, "non_mutation_suite_path"]

readiness_rows = []
for row in read_jsonl(MASTER_FILE):
    meta = row.get("metadata", {})
    readiness_rows.append({
        "n7": as_int(meta.get("benchmark_checks_passed")),
        "n8": as_int(meta.get("benchmark_checks_passed_v2")),
        "passed_n7": parse_list_like(meta.get("benchmark_passed_checks")),
        "passed_n8": parse_list_like(meta.get("benchmark_passed_checks_v2")),
    })

readiness = pd.DataFrame(readiness_rows)
n7_scores = readiness["n7"].dropna().astype(int)
n8_scores = readiness["n8"].dropna().astype(int)

def build_check_matrix(series: pd.Series, checks: list[str]) -> pd.DataFrame:
    return pd.DataFrame({check: series.apply(lambda values: check in set(values)).astype(float) for check in checks})

n8_matrix = build_check_matrix(readiness.loc[n8_scores.index, "passed_n8"], N8_CHECKS)
n8_observed = n8_matrix.sum(axis=1).astype(int)
n8_pass_rates = n8_matrix.mean(axis=0).to_numpy()
n8_expected = poisson_binomial_pmf(n8_pass_rates) * len(n8_matrix)
corr = n8_matrix.corr()

fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.5), constrained_layout=True)
ax = axes[0, 0]
counts = n7_scores.value_counts().reindex(range(8), fill_value=0)
ax.bar(counts.index, counts.values, color=COLORS["blue"], width=0.72)
ax.set_title("a  n/7 readiness scores")
ax.set_xlabel("checks passed")
ax.set_ylabel("rows")
ax.yaxis.set_major_formatter(FuncFormatter(thousands))
ax.set_xticks(range(8))

ax = axes[0, 1]
counts = n8_scores.value_counts().reindex(range(9), fill_value=0)
ax.bar(counts.index, counts.values, color=COLORS["green"], width=0.72)
ax.set_title("b  n/8 readiness scores")
ax.set_xlabel("checks passed")
ax.set_ylabel("rows")
ax.yaxis.set_major_formatter(FuncFormatter(thousands))
ax.set_xticks(range(9))

ax = axes[1, 0]
obs_counts = n8_observed.value_counts().reindex(range(9), fill_value=0)
x = np.arange(9)
ax.plot(x, obs_counts.values, marker="o", color=COLORS["green"], label="observed")
ax.plot(x, n8_expected, marker="s", color=COLORS["amber"], label="Poisson-binomial expected")
ax.set_title("c  observed vs null n/8 distribution")
ax.set_xlabel("checks passed")
ax.set_ylabel("rows")
ax.set_xticks(range(9))
ax.yaxis.set_major_formatter(FuncFormatter(thousands))
ax.legend(loc="upper left")

ax = axes[1, 1]
im = ax.imshow(corr.to_numpy(), vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_title("d  readiness-check correlation")
short_labels = [
    "valid", "extract", "no demo", "clean", "lines", "gates", "trusted", "non-mut",
]
ax.set_xticks(range(len(N8_CHECKS)), short_labels, rotation=45, ha="right")
ax.set_yticks(range(len(N8_CHECKS)), short_labels)
for i in range(len(N8_CHECKS)):
    for j in range(len(N8_CHECKS)):
        value = corr.iloc[i, j]
        if np.isfinite(value):
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="correlation")

save_and_display_figure(fig, "fig5_readiness_statistics")


## Figure 6: Semantic and Paraphrase Quality

This figure uses the canonical split files and paraphrase-diversity sidecar to summarize semantic consistency and lexical diversity.

In [ ]:
SPLIT_FILES = {
    "train": PROCESSED_DIR / "train_clean.jsonl",
    "validation": PROCESSED_DIR / "validation_clean.jsonl",
    "test": PROCESSED_DIR / "test_clean.jsonl",
}
SEMANTIC_FIELDS = [
    "bert_score_f1",
    "semantic_similarity_to_seed",
    "bleu_score_to_seed",
    "rouge_l_to_seed",
    "normalized_edit_distance",
]

metric_rows = []
for split, path in SPLIT_FILES.items():
    for row in read_jsonl(path):
        meta = row.get("metadata", {})
        is_paraphrase = bool(meta.get("original_prompt")) or meta.get("prompt_type") == "paraphrased_quality_aware"
        if not is_paraphrase:
            continue
        values = {field: meta.get(field) for field in SEMANTIC_FIELDS}
        if any(value not in (None, "") for value in values.values()):
            values["split"] = split
            metric_rows.append(values)

semantic = pd.DataFrame(metric_rows)
for field in SEMANTIC_FIELDS:
    semantic[field] = pd.to_numeric(semantic[field], errors="coerce")

diversity_file = PROCESSED_DIR / "paraphrase_diversity.jsonl"
diversity = pd.DataFrame(read_jsonl(diversity_file))
for field in ["bleu_mean", "bleu_min", "ttr_mean", "length_cv"]:
    if field in diversity:
        diversity[field] = pd.to_numeric(diversity[field], errors="coerce")

fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.5), constrained_layout=True)

ax = axes[0, 0]
ax.hist(semantic["bert_score_f1"].dropna(), bins=45, color=COLORS["blue"], alpha=0.9)
ax.set_title("a  BERTScore F1")
ax.set_xlabel("BERTScore F1")
ax.set_ylabel("paraphrase rows")
ax.yaxis.set_major_formatter(FuncFormatter(thousands))

ax = axes[0, 1]
ax.hist(semantic["semantic_similarity_to_seed"].dropna(), bins=45, color=COLORS["green"], alpha=0.9)
ax.set_title("b  sentence-transformer similarity")
ax.set_xlabel("cosine similarity to seed")
ax.set_ylabel("paraphrase rows")
ax.yaxis.set_major_formatter(FuncFormatter(thousands))

ax = axes[1, 0]
box_fields = ["bleu_score_to_seed", "rouge_l_to_seed", "normalized_edit_distance"]
box_data = [semantic[field].dropna().to_numpy() for field in box_fields]
bp = ax.boxplot(box_data, patch_artist=True, showfliers=False)
for patch, color in zip(bp["boxes"], [COLORS["amber"], COLORS["green"], COLORS["red"]]):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax.set_title("c  overlap and edit-distance metrics")
ax.set_xticks([1, 2, 3], ["BLEU", "ROUGE-L", "edit distance"])
ax.set_ylabel("score")

ax = axes[1, 1]
if not diversity.empty:
    ax.hist(diversity["bleu_mean"].dropna(), bins=35, color=COLORS["slate"], alpha=0.85)
    near_dup = int((diversity["bleu_min"] > 0.5).sum()) if "bleu_min" in diversity else 0
    total_groups = len(diversity)
    ax.axvline(0.5, color=COLORS["red"], linestyle="--", linewidth=1.2)
    ax.text(0.98, 0.95, f"near-duplicate groups\n{near_dup:,} / {total_groups:,}", transform=ax.transAxes, ha="right", va="top")
ax.set_title("d  group-level pairwise BLEU")
ax.set_xlabel("mean pairwise BLEU within group")
ax.set_ylabel("groups")
ax.yaxis.set_major_formatter(FuncFormatter(thousands))

save_and_display_figure(fig, "fig6_semantic_paraphrase_quality")


## Figure 7: License and Release Composition

This figure summarizes how the internal instruction object is partitioned into public and restricted release views.

In [ ]:
release_dir = PROCESSED_DIR / "release_views"
license_valid = json.loads((release_dir / "pqid_v1_license_valid_summary.json").read_text(encoding="utf-8"))
public_open = json.loads((release_dir / "pqid_v1_public_open_summary.json").read_text(encoding="utf-8"))

categories = ["permissive", "copyleft", "other", "no_license"]
category_labels = ["permissive", "copyleft", "reviewed other", "no license"]
category_colors = {
    "permissive": COLORS["green"],
    "copyleft": COLORS["amber"],
    "other": "#8b6fbd",
    "no_license": COLORS["red"],
}

internal_counts = Counter(license_valid["exported_license_category_counts"])
internal_counts.update(license_valid["excluded_license_category_counts"])
license_valid_counts = Counter(license_valid["exported_license_category_counts"])
public_open_counts = Counter(public_open["exported_license_category_counts"])
restricted_counts = Counter(license_valid["excluded_license_category_counts"])

fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.5), constrained_layout=True)

ax = axes[0, 0]
bar_names = ["internal", "license-valid", "public-open", "restricted"]
bar_maps = [internal_counts, license_valid_counts, public_open_counts, restricted_counts]
bottom = np.zeros(len(bar_names))
for category, label in zip(categories, category_labels):
    values = np.array([bar_map.get(category, 0) for bar_map in bar_maps])
    ax.bar(bar_names, values, bottom=bottom, color=category_colors[category], label=label)
    bottom += values
ax.set_title("a  release-view composition")
ax.set_ylabel("instruction rows")
ax.yaxis.set_major_formatter(FuncFormatter(thousands))
ax.tick_params(axis="x", rotation=20)
ax.legend(ncol=2, loc="upper right")

ax = axes[0, 1]
splits = ["train", "validation", "test"]
bottom = np.zeros(len(splits))
for category, label in zip(["permissive", "copyleft", "other"], ["permissive", "copyleft", "reviewed other"]):
    values = np.array([
        license_valid["splits"][split]["exported_license_category_counts"].get(category, 0)
        for split in splits
    ])
    ax.bar(splits, values, bottom=bottom, color=category_colors[category], label=label)
    bottom += values
ax.set_title("b  license-valid split composition")
ax.set_ylabel("exported rows")
ax.yaxis.set_major_formatter(FuncFormatter(thousands))
ax.legend(loc="upper right")

ax = axes[1, 0]
top_repos = license_valid.get("top_excluded_repositories", [])[:10]
repo_names = [repo["repo"].split("/")[-1] for repo in top_repos][::-1]
repo_values = [repo["excluded_rows"] for repo in top_repos][::-1]
ax.barh(repo_names, repo_values, color=COLORS["red"], alpha=0.85)
ax.set_title("c  largest restricted-source repositories")
ax.set_xlabel("excluded instruction rows")
ax.xaxis.set_major_formatter(FuncFormatter(thousands))

ax = axes[1, 1]
totals = {
    "internal": license_valid["total_input_rows"],
    "license-valid": license_valid["total_exported_rows"],
    "public-open": public_open["total_exported_rows"],
    "restricted": license_valid["total_excluded_rows"],
}
ax.bar(totals.keys(), totals.values(), color=[COLORS["slate"], COLORS["green"], COLORS["blue"], COLORS["red"]])
ax.set_title("d  release totals")
ax.set_ylabel("instruction rows")
ax.tick_params(axis="x", rotation=20)
ax.yaxis.set_major_formatter(FuncFormatter(thousands))
for idx, value in enumerate(totals.values()):
    ax.text(idx, value, f"{value:,}", ha="center", va="bottom", fontsize=8)

save_and_display_figure(fig, "fig7_release_composition")


## Supplementary Figure S4: Acquisition Pareto and Diminishing Returns

This supplementary figure quantifies source-acquisition concentration from the raw GitHub acquisition object. It uses repository-level Pareto coverage, concentration diagnostics (Gini coefficient and Herfindahl-Hirschman index), a descriptive log-log rank-yield decay fit, and rank-band marginal yield summaries to estimate how quickly additional repositories contribute fewer new rows after the highest-yield sources have been included. The fit is descriptive rather than causal, but it operationalizes the diminishing-returns pattern observed during acquisition.


In [ ]:
RAW_ACQUISITION_FILE = PROCESSED_DIR / "pqid_2026_raw_github_circuits.jsonl"

repo_counter = Counter()
strategy_counter = Counter()
run_counter = Counter()
for row in read_jsonl(RAW_ACQUISITION_FILE):
    meta = row.get("metadata", {})
    owner = meta.get("repo_owner")
    name = meta.get("repo_name")
    repo = f"{owner}/{name}" if owner and name else "<missing>"
    repo_counter[repo] += 1
    strategy_counter[meta.get("retrieval_strategy", "<missing>")] += 1
    run_counter[meta.get("retrieval_run_id", "<missing>")] += 1

repo_counts = pd.Series(repo_counter).sort_values(ascending=False)
repo_total = int(repo_counts.sum())
repo_n = int(len(repo_counts))
repo_ranks = np.arange(1, repo_n + 1)
repo_shares = repo_counts.to_numpy(dtype=float) / repo_total
repo_cumulative = np.cumsum(repo_shares)


def rank_for_share(target: float) -> int:
    return int(np.searchsorted(repo_cumulative, target) + 1)


def gini_coefficient(values: np.ndarray) -> float:
    values = np.sort(np.asarray(values, dtype=float))
    n = len(values)
    if n == 0 or values.sum() == 0:
        return float("nan")
    return float((2 * np.arange(1, n + 1).dot(values)) / (n * values.sum()) - (n + 1) / n)

rank50 = rank_for_share(0.50)
rank80 = rank_for_share(0.80)
rank95 = rank_for_share(0.95)
gini = gini_coefficient(repo_counts.to_numpy())
hhi = float((repo_shares ** 2).sum())

positive_counts = repo_counts[repo_counts > 0].to_numpy(dtype=float)
positive_ranks = np.arange(1, len(positive_counts) + 1, dtype=float)
log_rank = np.log10(positive_ranks)
log_yield = np.log10(positive_counts)
slope, intercept = np.polyfit(log_rank, log_yield, 1)
fit = slope * log_rank + intercept
rank_yield_r2 = 1 - float(((log_yield - fit) ** 2).sum() / ((log_yield - log_yield.mean()) ** 2).sum())

rank_bins = [
    (1, 5, "1-5"),
    (6, 25, "6-25"),
    (26, 100, "26-100"),
    (101, 500, "101-500"),
    (501, 1000, "501-1000"),
    (1001, repo_n, f"1001-{repo_n:,}"),
]
bin_labels = []
bin_means = []
bin_shares = []
for start, end, label in rank_bins:
    values = repo_counts.iloc[start - 1:end]
    if values.empty:
        continue
    bin_labels.append(label)
    bin_means.append(float(values.mean()))
    bin_shares.append(float(values.sum() / repo_total))

fig, axes = plt.subplots(2, 2, figsize=(10.8, 7.6), constrained_layout=True)

ax = axes[0, 0]
top_n = 15
top_repos = repo_counts.head(top_n)[::-1]
short_names = [name if len(name) <= 34 else "..." + name[-31:] for name in top_repos.index]
ax.barh(short_names, top_repos.values, color=COLORS["blue"], alpha=0.88)
ax.set_title("a  source-repository Pareto leaders")
ax.set_xlabel("raw acquisition rows")
ax.xaxis.set_major_formatter(FuncFormatter(thousands))

ax = axes[0, 1]
ax.plot(repo_ranks, repo_cumulative * 100, color=COLORS["green"], linewidth=2.0)
for target, rank, label in [(50, rank50, "50%"), (80, rank80, "80%"), (95, rank95, "95%")]:
    ax.axhline(target, color=COLORS["light"], linewidth=1.0)
    ax.axvline(rank, color=COLORS["slate"], linestyle="--", linewidth=0.9, alpha=0.75)
    ax.text(rank, target + 2.2, f"{label}: {rank:,} repos", fontsize=8, ha="left", va="bottom")
ax.set_title("b  Pareto coverage by repository rank")
ax.set_xlabel("repositories ranked by yield")
ax.set_ylabel("cumulative rows (%)")
ax.set_ylim(0, 103)
ax.set_xlim(1, repo_n)
ax.xaxis.set_major_formatter(FuncFormatter(thousands))

ax = axes[1, 0]
ax.scatter(positive_ranks, positive_counts, s=10, color=COLORS["slate"], alpha=0.35, linewidth=0)
ax.plot(positive_ranks, 10 ** fit, color=COLORS["red"], linewidth=1.8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("c  log-log rank-yield decay")
ax.set_xlabel("repository rank")
ax.set_ylabel("rows per repository")
ax.text(
    0.04,
    0.08,
    f"log-log slope = {slope:.2f}\n$R^2$ = {rank_yield_r2:.2f}",
    transform=ax.transAxes,
    fontsize=8,
    bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": COLORS["light"]},
)

ax = axes[1, 1]
bar_positions = np.arange(len(bin_labels))
ax.bar(bar_positions, bin_means, color=COLORS["amber"], alpha=0.88)
ax.set_yscale("log")
ax.set_title("d  diminishing marginal repository yield")
ax.set_xlabel("repository-rank band")
ax.set_ylabel("mean rows per repository")
ax.set_xticks(bar_positions, bin_labels, rotation=25, ha="right")
for idx, (mean_value, share) in enumerate(zip(bin_means, bin_shares)):
    ax.text(idx, mean_value, f"{share * 100:.1f}%", ha="center", va="bottom", fontsize=8)
ax.text(
    0.04,
    0.92,
    f"repositories = {repo_n:,}\nrows = {repo_total:,}\nGini = {gini:.2f}\nHHI = {hhi:.3f}",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": COLORS["light"]},
)

save_and_display_figure(fig, "suppfig_s4_acquisition_pareto_diminishing_returns")

acquisition_pareto_summary = {
    "raw_rows": repo_total,
    "unique_repositories": repo_n,
    "repositories_for_50_percent_rows": rank50,
    "repositories_for_80_percent_rows": rank80,
    "repositories_for_95_percent_rows": rank95,
    "gini_coefficient": round(gini, 4),
    "hhi": round(hhi, 4),
    "rank_yield_loglog_slope": round(float(slope), 4),
    "rank_yield_r2": round(rank_yield_r2, 4),
}
acquisition_pareto_summary


## Supplementary Figure S5: Linguistic Distribution and Audit Flow

This supplementary panel visualizes the language-audit sidecar used to scope PQID as an English-dominant instruction dataset. It keeps the compact distribution summaries and adds the branch-to-scope-to-resolution flow from the richer evidence-map diagnostic. The plot uses the compact audit summary plus the cached flow summary, rebuilding the cache only if the full language-audit sidecar changes.


In [ ]:
LANGUAGE_AUDIT_SUMMARY_FILE = PROCESSED_DIR / "instruction_language_audit_v1_summary.json"
LANGUAGE_AUDIT_FILE = PROCESSED_DIR / "instruction_language_audit_v1.jsonl"
LANGUAGE_AUDIT_FLOW_CACHE_FILE = PQID_ROOT / "submissions" / "scientific_data" / "LANGUAGE_AUDIT_FLOW_SUMMARY.json"

from collections import defaultdict
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Rectangle

language_summary = json.loads(LANGUAGE_AUDIT_SUMMARY_FILE.read_text(encoding="utf-8"))


def ordered_counter(mapping: dict[str, int], order: list[str] | None = None) -> pd.Series:
    counter = Counter(mapping)
    if order is None:
        items = sorted(counter.items(), key=lambda item: item[1], reverse=True)
    else:
        items = [(key, counter.get(key, 0)) for key in order if counter.get(key, 0)]
    return pd.Series(dict(items), dtype="int64")


def pct_label(value: int, total: int) -> str:
    pct = value / total * 100 if total else 0
    if pct > 99.9 or pct < 0.1:
        return f"{value:,}\n{pct:.4f}%"
    return f"{value:,}\n{pct:.2f}%"


def language_group(label: str) -> str:
    if label == "en":
        return "english"
    if label == "none":
        return "code_only"
    return "tail"


def key_join(*parts: str) -> str:
    return "|||".join(parts)


def unpack_key(key: str) -> tuple[str, ...]:
    return tuple(key.split("|||"))


def audit_file_signature(path: Path) -> dict[str, int | str | bool]:
    if not path.exists():
        return {"path": str(path), "missing": True}
    stat = path.stat()
    return {"path": str(path), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}


def load_language_flow_summary() -> dict:
    signature = audit_file_signature(LANGUAGE_AUDIT_FILE)
    if LANGUAGE_AUDIT_FLOW_CACHE_FILE.exists():
        cached = json.loads(LANGUAGE_AUDIT_FLOW_CACHE_FILE.read_text(encoding="utf-8"))
        if signature.get("missing") or cached.get("source_signature") == signature:
            return cached
    if signature.get("missing"):
        raise FileNotFoundError(f"Missing language-audit sidecar and no usable cache: {LANGUAGE_AUDIT_FILE}")

    branch_scope = Counter()
    scope_resolution_group = Counter()
    rows = 0
    for row in read_jsonl(LANGUAGE_AUDIT_FILE):
        rows += 1
        branch = str(row.get("source_branch") or "<missing>")
        scope = str(row.get("output_human_language_scope") or "<missing>")
        resolution = str(row.get("output_human_language_resolved") or "<missing>")
        group = language_group(resolution)
        branch_scope[key_join(branch, scope)] += 1
        scope_resolution_group[key_join(scope, group)] += 1

    result = {
        "summary_version": "language_audit_flow_summary_v1",
        "source_signature": signature,
        "rows": rows,
        "branch_scope_counts": dict(branch_scope),
        "scope_resolution_group_counts": dict(scope_resolution_group),
    }
    LANGUAGE_AUDIT_FLOW_CACHE_FILE.write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


def stack_positions(labels, visual_totals, y_top=0.88, y_bottom=0.12, gap=0.028):
    total_visual = sum(max(visual_totals.get(label, 0), 0) for label in labels)
    available = y_top - y_bottom - gap * (len(labels) - 1)
    positions = {}
    cursor = y_top
    for label in labels:
        height = available * visual_totals.get(label, 0) / total_visual if total_visual else 0
        positions[label] = (cursor - height, cursor)
        cursor -= height + gap
    return positions


def allocate_segments(node_positions, labels, flow_keys, source_index, target_index, counts):
    by_node = defaultdict(list)
    for key in flow_keys:
        parts = unpack_key(key)
        by_node[parts[source_index]].append(key)
        by_node[parts[target_index]].append(key)

    segments = {}
    cursors = {label: node_positions[label][0] for label in labels}
    for label in labels:
        node_low, node_high = node_positions[label]
        node_height = node_high - node_low
        keys = sorted(set(by_node[label]), key=lambda k: unpack_key(k))
        visual_total = sum(
            np.sqrt(counts[k])
            for k in keys
            if label in (unpack_key(k)[source_index], unpack_key(k)[target_index])
        )
        for key in keys:
            parts = unpack_key(key)
            if label not in (parts[source_index], parts[target_index]):
                continue
            height = node_height * np.sqrt(counts[key]) / visual_total if visual_total else 0
            segments[(label, key)] = (cursors[label], cursors[label] + height)
            cursors[label] += height
    return segments


def draw_ribbon(ax, x0, y0_low, y0_high, x1, y1_low, y1_high, color, alpha=0.35):
    dx = x1 - x0
    c0 = x0 + dx * 0.44
    c1 = x1 - dx * 0.44
    verts = [
        (x0, y0_high), (c0, y0_high), (c1, y1_high), (x1, y1_high),
        (x1, y1_low), (c1, y1_low), (c0, y0_low), (x0, y0_low), (x0, y0_high),
    ]
    codes = [
        MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
        MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
        MplPath.CLOSEPOLY,
    ]
    ax.add_patch(PathPatch(MplPath(verts, codes), facecolor=color, edgecolor="none", alpha=alpha))


flow_summary = load_language_flow_summary()

input_lang = ordered_counter(language_summary["input_human_language_resolved_counts"], ["en", "bn"])
output_scope = ordered_counter(
    language_summary["output_human_language_scope_counts"],
    ["full_output_text", "code_only", "code_comments_or_docstrings"],
)
output_tail = ordered_counter(language_summary["output_human_language_resolved_counts"])
output_tail = output_tail.drop(labels=[label for label in ["en", "none"] if label in output_tail.index])
script_tail = ordered_counter(language_summary["output_human_script_bucket_counts"])
script_tail = script_tail.drop(labels=[label for label in ["latin"] if label in script_tail.index])

lang_labels = {
    "en": "English",
    "bn": "Bengali",
    "es": "Spanish",
    "pt": "Portuguese",
    "fr": "French",
    "ja_script": "Japanese script",
    "ko_script": "Korean script",
    "mixed": "mixed",
    "short_fragment": "short fragment",
    "cyrillic_script_unresolved": "Cyrillic unresolved",
    "none": "none",
}
scope_labels = {
    "full_output_text": "full generated text",
    "code_only": "code only",
    "code_comments_or_docstrings": "comments/docstrings",
}
script_labels = {
    "latin_plus_greek": "Latin + Greek",
    "mixed_scripts": "mixed scripts",
    "hangul": "Hangul",
    "none": "none",
    "latin_plus_hangul": "Latin + Hangul",
    "latin_plus_cyrillic": "Latin + Cyrillic",
    "latin_plus_cjk": "Latin + CJK",
    "latin_plus_arabic": "Latin + Arabic",
}
branch_labels = {
    "teacher_text": "teacher-text",
    "source_code": "source-code",
}
resolution_group_labels = {
    "english": "English text",
    "code_only": "code-only / no human text",
    "tail": "non-English / ambiguous tail",
}

fig = plt.figure(figsize=(10.8, 9.15), constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.035, h_pad=0.035, hspace=0.06, wspace=0.07)
gs = fig.add_gridspec(3, 2, height_ratios=[0.95, 1.35, 1.08])
ax_a = fig.add_subplot(gs[0, 0])
ax_c = fig.add_subplot(gs[0, 1])
ax_b = fig.add_subplot(gs[1, :])
ax_d = fig.add_subplot(gs[2, 0])
ax_e = fig.add_subplot(gs[2, 1])

# A. Input instruction language.
input_total = int(input_lang.sum())
colors = [COLORS["green"] if label == "en" else COLORS["amber"] for label in input_lang.index]
y = np.arange(len(input_lang))[::-1]
ax_a.barh(y, input_lang.values, color=colors, alpha=0.88)
ax_a.set_yticks(y, [lang_labels.get(label, label) for label in input_lang.index])
ax_a.set_xscale("log")
ax_a.set_title("a  input instruction language")
ax_a.set_xlabel("instruction rows, log scale")
ax_a.xaxis.set_major_formatter(FuncFormatter(thousands))
for ypos, value in zip(y, input_lang.values):
    ax_a.text(value * 1.08, ypos, pct_label(int(value), input_total), va="center", fontsize=8)
ax_a.text(
    0.03,
    0.08,
    f"resolved rows = {input_total:,}",
    transform=ax_a.transAxes,
    fontsize=8,
    bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": COLORS["light"]},
)

# B. Branch-to-scope-to-resolution flow.
ax_b.set_title("b  branch -> output scope -> resolved output class")
ax_b.set_xlim(0, 1)
ax_b.set_ylim(0, 1)
ax_b.axis("off")
branch_order = ["teacher_text", "source_code"]
scope_order = ["full_output_text", "code_comments_or_docstrings", "code_only"]
group_order = ["english", "tail", "code_only"]
branch_scope_counts = Counter(flow_summary["branch_scope_counts"])
scope_group_counts = Counter(flow_summary["scope_resolution_group_counts"])
branch_visual = {branch: sum(np.sqrt(branch_scope_counts.get(key_join(branch, scope), 0)) for scope in scope_order) for branch in branch_order}
scope_visual = {
    scope: max(
        sum(np.sqrt(branch_scope_counts.get(key_join(branch, scope), 0)) for branch in branch_order),
        sum(np.sqrt(scope_group_counts.get(key_join(scope, group), 0)) for group in group_order),
    )
    for scope in scope_order
}
group_visual = {group: sum(np.sqrt(scope_group_counts.get(key_join(scope, group), 0)) for scope in scope_order) for group in group_order}
branch_pos = stack_positions(branch_order, branch_visual, y_top=0.86, y_bottom=0.16, gap=0.04)
scope_pos = stack_positions(scope_order, scope_visual, y_top=0.88, y_bottom=0.14, gap=0.035)
group_pos = stack_positions(group_order, group_visual, y_top=0.88, y_bottom=0.14, gap=0.035)
x_branch, x_scope, x_group = 0.11, 0.50, 0.84
node_w = 0.028

bs_keys = [key_join(branch, scope) for branch in branch_order for scope in scope_order if branch_scope_counts.get(key_join(branch, scope), 0)]
bs_segments = allocate_segments({**branch_pos, **scope_pos}, branch_order + scope_order, bs_keys, 0, 1, branch_scope_counts)
for key in bs_keys:
    branch, scope = unpack_key(key)
    color = COLORS["blue"] if branch == "teacher_text" else COLORS["green"]
    draw_ribbon(ax_b, x_branch + node_w / 2, *bs_segments[(branch, key)], x_scope - node_w / 2, *bs_segments[(scope, key)], color, alpha=0.27)
sg_keys = [key_join(scope, group) for scope in scope_order for group in group_order if scope_group_counts.get(key_join(scope, group), 0)]
sg_segments = allocate_segments({**scope_pos, **group_pos}, scope_order + group_order, sg_keys, 0, 1, scope_group_counts)
for key in sg_keys:
    scope, group = unpack_key(key)
    color = COLORS["green"] if group == "english" else COLORS["red"] if group == "tail" else COLORS["slate"]
    draw_ribbon(ax_b, x_scope + node_w / 2, *sg_segments[(scope, key)], x_group - node_w / 2, *sg_segments[(group, key)], color, alpha=0.31)

for branch in branch_order:
    low, high = branch_pos[branch]
    count = language_summary["branch_counts"].get(branch, 0)
    color = COLORS["blue"] if branch == "teacher_text" else COLORS["green"]
    ax_b.add_patch(Rectangle((x_branch - node_w / 2, low), node_w, high - low, color=color, alpha=0.95, zorder=3))
    ax_b.text(x_branch - 0.030, (low + high) / 2, f"{branch_labels[branch]}\n{count:,}", ha="right", va="center", fontsize=8.0, zorder=4)
for scope in scope_order:
    low, high = scope_pos[scope]
    count = language_summary["output_human_language_scope_counts"].get(scope, 0)
    color = COLORS["amber"] if scope == "code_comments_or_docstrings" else COLORS["slate"] if scope == "code_only" else COLORS["blue"]
    ax_b.add_patch(Rectangle((x_scope - node_w / 2, low), node_w, high - low, color=color, alpha=0.92, zorder=3))
    label_y = min(high + 0.018, 0.96)
    ax_b.text(x_scope, label_y, f"{scope_labels[scope]}\n{count:,}", ha="center", va="bottom", fontsize=7.1, zorder=4)
for group in group_order:
    low, high = group_pos[group]
    count = sum(scope_group_counts.get(key_join(scope, group), 0) for scope in scope_order)
    color = COLORS["green"] if group == "english" else COLORS["red"] if group == "tail" else COLORS["slate"]
    ax_b.add_patch(Rectangle((x_group - node_w / 2, low), node_w, high - low, color=color, alpha=0.95, zorder=3))
    ax_b.text(x_group + 0.028, (low + high) / 2, f"{resolution_group_labels[group]}\n{count:,}", ha="left", va="center", fontsize=7.6, zorder=4)
ax_b.text(0.03, 0.055, "ribbon heights use sqrt(count) scaling; exact counts are printed", fontsize=7.0, color=COLORS["slate"])

# C. Output language-audit scope.
scope_ordered = output_scope.sort_values(ascending=False)
bar_colors = [COLORS["blue"], COLORS["slate"], COLORS["amber"]]
ax_c.bar([scope_labels.get(label, label) for label in scope_ordered.index], scope_ordered.values, color=bar_colors, alpha=0.88)
ax_c.set_title("c  output language-audit scope")
ax_c.set_ylabel("instruction rows")
ax_c.yaxis.set_major_formatter(FuncFormatter(thousands))
ax_c.tick_params(axis="x", rotation=18)
for idx, value in enumerate(scope_ordered.values):
    ax_c.text(idx, value, f"{int(value):,}", ha="center", va="bottom", fontsize=8)

# D. Resolved non-English or ambiguous output tail.
tail = output_tail.sort_values(ascending=True)
ax_d.barh([lang_labels.get(label, label) for label in tail.index], tail.values, color=COLORS["red"], alpha=0.78)
ax_d.set_title("d  resolved non-English / ambiguous output tail")
ax_d.set_xlabel("rows")
ax_d.xaxis.set_major_formatter(FuncFormatter(thousands))
for idx, value in enumerate(tail.values):
    ax_d.text(value + max(tail.values) * 0.015, idx, f"{int(value):,}", va="center", fontsize=8)

# E. Non-Latin and mixed script buckets.
script = script_tail.sort_values(ascending=True)
ax_e.barh([script_labels.get(label, label) for label in script.index], script.values, color="#8b6fbd", alpha=0.82)
ax_e.set_title("e  non-Latin or mixed script buckets in outputs")
ax_e.set_xlabel("rows")
ax_e.xaxis.set_major_formatter(FuncFormatter(thousands))
for idx, value in enumerate(script.values):
    ax_e.text(value + max(script.values) * 0.015, idx, f"{int(value):,}", va="center", fontsize=8)

for ax in [ax_a, ax_c, ax_d, ax_e]:
    ax.grid(axis="x", color=COLORS["light"], linewidth=0.8)

save_and_display_figure(fig, "suppfig_s5_linguistic_distribution")

linguistic_distribution_summary = {
    "rows": int(language_summary["rows"]),
    "flow_rows": int(flow_summary["rows"]),
    "input_human_language_resolved_counts": {str(k): int(v) for k, v in input_lang.items()},
    "output_human_language_scope_counts": {str(k): int(v) for k, v in output_scope.items()},
    "output_human_language_resolved_tail_counts": {str(k): int(v) for k, v in output_tail.items()},
    "output_human_script_tail_counts": {str(k): int(v) for k, v in script_tail.items()},
}
display(Markdown("**Linguistic distribution and audit-flow summary**\n\n```json\n" + json.dumps(linguistic_distribution_summary, indent=2) + "\n```"))


## Supplementary Figure S6: License-Behaviour and Obligation Clustering

This supplementary panel renders the detected-license behavioural clustering used to interpret release obligations. The plotting logic is kept in `plot_license_behavior_panel.py` so that the figure, the Calibri export, and the two supporting CSV tables can be regenerated from the same source.


In [ ]:
from IPython.display import Image as IPythonImage
import runpy

SCI_DATA_DIR = PQID_ROOT / "submissions" / "scientific_data"
S6_SCRIPT = SCI_DATA_DIR / "plot_license_behavior_panel.py"
S6_CANONICAL_PNG = SCI_DATA_DIR / "figures" / "suppfig_s6_license_behavior_panel.png"
S6_CALIBRI_PNG = SCI_DATA_DIR / "figures_calibri" / "suppfig_s6_license_behavior_panel.png"
S6_BEHAVIOR_TABLE = SCI_DATA_DIR / "LICENSE_BEHAVIOR_DISTRIBUTION.csv"
S6_RELEASE_VIEW_TABLE = SCI_DATA_DIR / "LICENSE_BEHAVIOR_RELEASE_VIEW_DISTRIBUTION.csv"

assert S6_SCRIPT.exists(), f"Missing S6 plotting script: {S6_SCRIPT}"
runpy.run_path(str(S6_SCRIPT), run_name="__main__")

display(Markdown(
    "**Supplementary Figure S6 regenerated**  \n"
    f"- Canonical export: `{S6_CANONICAL_PNG}`  \n"
    f"- Calibri export: `{S6_CALIBRI_PNG}`  \n"
    f"- Behaviour table: `{S6_BEHAVIOR_TABLE}`  \n"
    f"- Release-view table: `{S6_RELEASE_VIEW_TABLE}`"
))
display(IPythonImage(filename=str(S6_CALIBRI_PNG)))

display(Markdown("**S6 behavioural-family distribution**"))
display(pd.read_csv(S6_BEHAVIOR_TABLE))
display(Markdown("**S6 release-view by behavioural-family distribution**"))
display(pd.read_csv(S6_RELEASE_VIEW_TABLE))


## Backup Diagnostic: Language-Audit Evidence Map

This larger diagnostic is retained as an optional audit view. The preferred supplementary figure is now S5, which combines the clearer distribution panels with the branch-to-scope-to-resolution flow.


In [ ]:
LANGUAGE_AUDIT_SUMMARY_FILE = PROCESSED_DIR / "instruction_language_audit_v1_summary.json"
LANGUAGE_AUDIT_FILE = PROCESSED_DIR / "instruction_language_audit_v1.jsonl"
LANGUAGE_AUDIT_FLOW_CACHE_FILE = PQID_ROOT / "submissions" / "scientific_data" / "LANGUAGE_AUDIT_FLOW_SUMMARY.json"

from collections import defaultdict
import textwrap
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Rectangle, FancyBboxPatch

language_summary = json.loads(LANGUAGE_AUDIT_SUMMARY_FILE.read_text(encoding="utf-8"))

lang_labels = {
    "en": "English",
    "bn": "Bengali",
    "es": "Spanish",
    "pt": "Portuguese",
    "fr": "French",
    "ja_script": "Japanese script",
    "ko_script": "Korean script",
    "mixed": "mixed",
    "short_fragment": "short fragment",
    "cyrillic_script_unresolved": "Cyrillic unresolved",
    "none": "code-only / none",
}
scope_labels = {
    "full_output_text": "full generated text",
    "code_only": "code-only output",
    "code_comments_or_docstrings": "comments/docstrings",
}
branch_labels = {
    "teacher_text": "teacher-text",
    "source_code": "source-code",
}
resolution_group_labels = {
    "english": "English text",
    "code_only": "code-only / no human text",
    "tail": "non-English / ambiguous tail",
}


def language_group(label: str) -> str:
    if label == "en":
        return "english"
    if label == "none":
        return "code_only"
    return "tail"


def key_join(*parts: str) -> str:
    return "|||".join(parts)


def unpack_key(key: str) -> tuple[str, ...]:
    return tuple(key.split("|||"))


def audit_file_signature(path: Path) -> dict[str, int | str]:
    stat = path.stat()
    return {"path": str(path), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}


def load_language_flow_summary() -> dict:
    signature = audit_file_signature(LANGUAGE_AUDIT_FILE)
    if LANGUAGE_AUDIT_FLOW_CACHE_FILE.exists():
        cached = json.loads(LANGUAGE_AUDIT_FLOW_CACHE_FILE.read_text(encoding="utf-8"))
        if cached.get("source_signature") == signature:
            return cached

    branch_scope = Counter()
    scope_resolution_group = Counter()
    branch_resolution = Counter()
    branch_scope_resolution = Counter()
    scope_resolution = Counter()
    rows = 0
    for row in read_jsonl(LANGUAGE_AUDIT_FILE):
        rows += 1
        branch = str(row.get("source_branch") or "<missing>")
        scope = str(row.get("output_human_language_scope") or "<missing>")
        resolution = str(row.get("output_human_language_resolved") or "<missing>")
        group = language_group(resolution)
        branch_scope[key_join(branch, scope)] += 1
        scope_resolution_group[key_join(scope, group)] += 1
        branch_resolution[key_join(branch, resolution)] += 1
        branch_scope_resolution[key_join(branch, scope, resolution)] += 1
        scope_resolution[key_join(scope, resolution)] += 1

    result = {
        "summary_version": "language_audit_flow_summary_v1",
        "source_signature": signature,
        "rows": rows,
        "branch_scope_counts": dict(branch_scope),
        "scope_resolution_group_counts": dict(scope_resolution_group),
        "branch_resolution_counts": dict(branch_resolution),
        "branch_scope_resolution_counts": dict(branch_scope_resolution),
        "scope_resolution_counts": dict(scope_resolution),
    }
    LANGUAGE_AUDIT_FLOW_CACHE_FILE.write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


flow_summary = load_language_flow_summary()


def ordered_counter(mapping: dict[str, int], order: list[str] | None = None) -> pd.Series:
    counter = Counter(mapping)
    if order is None:
        items = sorted(counter.items(), key=lambda item: item[1], reverse=True)
    else:
        items = [(key, counter.get(key, 0)) for key in order if counter.get(key, 0)]
    return pd.Series(dict(items), dtype="int64")


def pct_text(value: int, total: int, digits: int = 2) -> str:
    pct = value / total * 100 if total else 0
    return f"{pct:.{digits}f}%"


def rounded_box(ax, xy, width, height, color, alpha=1.0, radius=0.03, ec="none", lw=0.8):
    patch = FancyBboxPatch(
        xy,
        width,
        height,
        boxstyle=f"round,pad=0.006,rounding_size={radius}",
        linewidth=lw,
        edgecolor=ec,
        facecolor=color,
        alpha=alpha,
    )
    ax.add_patch(patch)
    return patch


def stack_positions(labels, visual_totals, y_top=0.91, y_bottom=0.09, gap=0.035):
    total_visual = sum(max(visual_totals.get(label, 0), 0) for label in labels)
    available = y_top - y_bottom - gap * (len(labels) - 1)
    positions = {}
    cursor = y_top
    for label in labels:
        height = available * visual_totals.get(label, 0) / total_visual if total_visual else 0
        positions[label] = (cursor - height, cursor)
        cursor -= height + gap
    return positions


def allocate_segments(node_positions, labels, flow_keys, source_index, target_index, counts):
    by_node = defaultdict(list)
    for key in flow_keys:
        parts = unpack_key(key)
        by_node[parts[source_index]].append(key)
        by_node[parts[target_index]].append(key)

    incoming_or_outgoing = defaultdict(float)
    for key in flow_keys:
        value = np.sqrt(counts[key])
        src, dst = unpack_key(key)[source_index], unpack_key(key)[target_index]
        incoming_or_outgoing[src] += value
        incoming_or_outgoing[dst] += value

    segments = {}
    cursors = {label: node_positions[label][0] for label in labels}
    for label in labels:
        node_low, node_high = node_positions[label]
        node_height = node_high - node_low
        keys = sorted(set(by_node[label]), key=lambda k: unpack_key(k))
        visual_total = sum(np.sqrt(counts[k]) for k in keys if label in (unpack_key(k)[source_index], unpack_key(k)[target_index]))
        for key in keys:
            parts = unpack_key(key)
            if label not in (parts[source_index], parts[target_index]):
                continue
            height = node_height * np.sqrt(counts[key]) / visual_total if visual_total else 0
            segments[(label, key)] = (cursors[label], cursors[label] + height)
            cursors[label] += height
    return segments


def draw_ribbon(ax, x0, y0_low, y0_high, x1, y1_low, y1_high, color, alpha=0.42):
    dx = x1 - x0
    c0 = x0 + dx * 0.45
    c1 = x1 - dx * 0.45
    verts = [
        (x0, y0_high),
        (c0, y0_high),
        (c1, y1_high),
        (x1, y1_high),
        (x1, y1_low),
        (c1, y1_low),
        (c0, y0_low),
        (x0, y0_low),
        (x0, y0_high),
    ]
    codes = [
        MplPath.MOVETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CLOSEPOLY,
    ]
    ax.add_patch(PathPatch(MplPath(verts, codes), facecolor=color, edgecolor="none", alpha=alpha))


fig, axes = plt.subplots(2, 2, figsize=(11.4, 8.15), constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.04, h_pad=0.04, hspace=0.08, wspace=0.08)

# A. Input-language dominance plus script tail.
ax = axes[0, 0]
ax.set_title("a  input language and script evidence")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
input_lang = ordered_counter(language_summary["input_human_language_resolved_counts"], ["en", "bn"])
input_total = int(input_lang.sum())
english_count = int(input_lang.get("en", 0))
bengali_count = int(input_lang.get("bn", 0))
strip_x, strip_y, strip_w, strip_h = 0.05, 0.62, 0.88, 0.12
rounded_box(ax, (strip_x, strip_y), strip_w, strip_h, COLORS["light"], radius=0.025)
ax.add_patch(Rectangle((strip_x, strip_y), strip_w * english_count / input_total, strip_h, color=COLORS["green"], alpha=0.9, linewidth=0))
tail_x = strip_x + strip_w - 0.004
ax.add_patch(Rectangle((tail_x, strip_y - 0.012), 0.006, strip_h + 0.024, color=COLORS["amber"], alpha=0.95, linewidth=0))
ax.text(strip_x + 0.025, strip_y + strip_h / 2, f"English  {english_count:,} ({pct_text(english_count, input_total, 4)})", va="center", ha="left", color="white", fontsize=9.2, weight="bold")
rounded_box(ax, (0.55, 0.35), 0.34, 0.14, COLORS["amber"], alpha=0.95, radius=0.025)
ax.text(0.72, 0.42, f"Bengali tail\n{bengali_count:,} rows ({pct_text(bengali_count, input_total, 4)})", va="center", ha="center", color="white", fontsize=8.7, weight="bold")
ax.plot([tail_x + 0.004, 0.72], [strip_y - 0.01, 0.49], color=COLORS["amber"], linewidth=1.0, linestyle="--")
ax.text(0.05, 0.79, f"resolved instruction rows: {input_total:,}", fontsize=8.4, color=COLORS["slate"])
script_counts = ordered_counter(language_summary["input_human_script_bucket_counts"])
script_tail = script_counts.drop(labels=[label for label in ["latin"] if label in script_counts.index])
script_labels = {
    "latin_plus_greek": "Latin+Greek",
    "latin_plus_devanagari": "Latin+Devanagari",
    "latin_plus_hebrew": "Latin+Hebrew",
    "latin_plus_bengali": "Latin+Bengali",
    "latin_plus_arabic": "Latin+Arabic",
    "mixed_scripts": "mixed",
}
ax.text(0.05, 0.245, "input script tail", fontsize=8.5, color=COLORS["slate"], weight="bold")
x0, x1, y0 = 0.05, 0.92, 0.18
max_script = max(script_tail.values)
for idx, (label, value) in enumerate(script_tail.items()):
    x = x0 + (x1 - x0) * idx / max(1, len(script_tail) - 1)
    size = 32 + 180 * np.sqrt(value / max_script)
    ax.scatter([x], [y0], s=size, color="#8b6fbd", alpha=0.78, edgecolor="white", linewidth=0.8)
    ax.text(x, y0 - 0.065, f"{script_labels.get(label, label)}\n{int(value):,}", ha="center", va="top", fontsize=6.7, color=COLORS["slate"])

# B. Alluvial evidence flow.
ax = axes[0, 1]
ax.set_title("b  branch -> output scope -> resolved output class")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
branch_order = ["teacher_text", "source_code"]
scope_order = ["full_output_text", "code_comments_or_docstrings", "code_only"]
group_order = ["english", "tail", "code_only"]
branch_scope_counts = Counter(flow_summary["branch_scope_counts"])
scope_group_counts = Counter(flow_summary["scope_resolution_group_counts"])
branch_visual = {branch: sum(np.sqrt(branch_scope_counts.get(key_join(branch, scope), 0)) for scope in scope_order) for branch in branch_order}
scope_visual = {
    scope: max(
        sum(np.sqrt(branch_scope_counts.get(key_join(branch, scope), 0)) for branch in branch_order),
        sum(np.sqrt(scope_group_counts.get(key_join(scope, group), 0)) for group in group_order),
    )
    for scope in scope_order
}
group_visual = {group: sum(np.sqrt(scope_group_counts.get(key_join(scope, group), 0)) for scope in scope_order) for group in group_order}
branch_pos = stack_positions(branch_order, branch_visual)
scope_pos = stack_positions(scope_order, scope_visual)
group_pos = stack_positions(group_order, group_visual)
x_branch, x_scope, x_group = 0.08, 0.48, 0.88
node_w = 0.035
for branch in branch_order:
    low, high = branch_pos[branch]
    count = language_summary["branch_counts"].get(branch, 0)
    color = COLORS["blue"] if branch == "teacher_text" else COLORS["green"]
    ax.add_patch(Rectangle((x_branch - node_w / 2, low), node_w, high - low, color=color, alpha=0.9))
    ax.text(x_branch - 0.035, (low + high) / 2, f"{branch_labels[branch]}\n{count:,}", ha="right", va="center", fontsize=7.7)
for scope in scope_order:
    low, high = scope_pos[scope]
    count = language_summary["output_human_language_scope_counts"].get(scope, 0)
    color = COLORS["amber"] if scope == "code_comments_or_docstrings" else COLORS["slate"] if scope == "code_only" else COLORS["blue"]
    ax.add_patch(Rectangle((x_scope - node_w / 2, low), node_w, high - low, color=color, alpha=0.85))
    ax.text(x_scope, high + 0.014, f"{scope_labels[scope]}\n{count:,}", ha="center", va="bottom", fontsize=6.8)
for group in group_order:
    low, high = group_pos[group]
    count = sum(scope_group_counts.get(key_join(scope, group), 0) for scope in scope_order)
    color = COLORS["green"] if group == "english" else COLORS["red"] if group == "tail" else COLORS["slate"]
    ax.add_patch(Rectangle((x_group - node_w / 2, low), node_w, high - low, color=color, alpha=0.9))
    ax.text(x_group + 0.035, (low + high) / 2, f"{resolution_group_labels[group]}\n{count:,}", ha="left", va="center", fontsize=7.2)

bs_keys = [key_join(branch, scope) for branch in branch_order for scope in scope_order if branch_scope_counts.get(key_join(branch, scope), 0)]
bs_segments = allocate_segments({**branch_pos, **scope_pos}, branch_order + scope_order, bs_keys, 0, 1, branch_scope_counts)
for key in bs_keys:
    branch, scope = unpack_key(key)
    color = COLORS["blue"] if branch == "teacher_text" else COLORS["green"]
    draw_ribbon(ax, x_branch + node_w / 2, *bs_segments[(branch, key)], x_scope - node_w / 2, *bs_segments[(scope, key)], color, alpha=0.26)
sg_keys = [key_join(scope, group) for scope in scope_order for group in group_order if scope_group_counts.get(key_join(scope, group), 0)]
sg_segments = allocate_segments({**scope_pos, **group_pos}, scope_order + group_order, sg_keys, 0, 1, scope_group_counts)
for key in sg_keys:
    scope, group = unpack_key(key)
    color = COLORS["green"] if group == "english" else COLORS["red"] if group == "tail" else COLORS["slate"]
    draw_ribbon(ax, x_scope + node_w / 2, *sg_segments[(scope, key)], x_group - node_w / 2, *sg_segments[(group, key)], color, alpha=0.31)
ax.text(0.05, 0.035, "ribbon heights use sqrt(count) scaling; exact counts are printed", fontsize=6.8, color=COLORS["slate"])

# C. Branch-conditioned resolved-language heatmap.
ax = axes[1, 0]
heat_rows = ["en", "none", "short_fragment", "es", "ja_script", "pt", "mixed", "ko_script", "fr", "cyrillic_script_unresolved"]
heat_cols = ["source_code", "teacher_text"]
branch_resolved = language_summary["branch_output_human_language_resolved_counts"]
mat = np.array([[branch_resolved.get(col, {}).get(row, 0) for col in heat_cols] for row in heat_rows], dtype=float)
log_mat = np.log10(mat + 1)
cmap = LinearSegmentedColormap.from_list("pqid_language_heat", ["#f7fafc", "#cfe3f4", COLORS["blue"], COLORS["slate"]])
im = ax.imshow(log_mat, aspect="auto", cmap=cmap)
ax.set_title("c  branch-conditioned resolved output language")
ax.set_xticks(np.arange(len(heat_cols)), [branch_labels[col] for col in heat_cols])
ax.set_yticks(np.arange(len(heat_rows)), [lang_labels.get(row, row) for row in heat_rows])
for i in range(len(heat_rows)):
    for j in range(len(heat_cols)):
        value = int(mat[i, j])
        if value:
            color = "white" if log_mat[i, j] > 4.3 else "#111111"
            ax.text(j, i, f"{value:,}", ha="center", va="center", fontsize=7.3, color=color)
ax.set_xticks(np.arange(-.5, len(heat_cols), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(heat_rows), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1.4)
ax.tick_params(which="minor", bottom=False, left=False)
cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
cb.set_label("log10(rows + 1)", fontsize=7.5)
cb.ax.tick_params(labelsize=7)

# D. Compact example evidence panel.
ax = axes[1, 1]
ax.set_title("d  representative audit examples")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
examples = language_summary.get("non_english_examples", [])
wanted = ["short_fragment", "es", "pt", "fr", "ja_script", "ko_script"]
selected = []
seen = set()
for target in wanted:
    for ex in examples:
        label = ex.get("output_human_language_resolved")
        if label == target and label not in seen:
            selected.append(ex)
            seen.add(label)
            break
for ex in examples:
    if len(selected) >= 6:
        break
    label = ex.get("output_human_language_resolved")
    if label not in seen:
        selected.append(ex)
        seen.add(label)

signal_map = {
    "short_fragment": "short fragment",
    "es": "Spanish comment",
    "pt": "Portuguese comment",
    "fr": "French comment",
    "ja_script": "Japanese-script comment",
    "ko_script": "Korean-script comment",
    "mixed": "mixed-language evidence",
    "cyrillic_script_unresolved": "Cyrillic unresolved",
}
lefts = [0.02, 0.27, 0.56, 0.96]
headers = ["resolved label", "scope", "audit signal", "key"]
for x, h in zip(lefts, headers):
    ha = "right" if h == "key" else "left"
    ax.text(x, 0.92, h, fontsize=7.4, weight="bold", color=COLORS["slate"], va="top", ha=ha)
ax.plot([0.02, 0.98], [0.885, 0.885], color=COLORS["light"], linewidth=1.2)
row_h = 0.115
for idx, ex in enumerate(selected):
    y_top = 0.86 - idx * row_h
    y_mid = y_top - row_h * 0.46
    if idx % 2 == 0:
        ax.add_patch(Rectangle((0.015, y_top - row_h + 0.014), 0.965, row_h - 0.012, facecolor="#f7fafc", edgecolor="none"))
    raw_label = ex.get("output_human_language_resolved")
    label = lang_labels.get(raw_label, raw_label)
    scope = {"code_comments_or_docstrings": "comment/docstring", "code_only": "code-only", "full_output_text": "full text"}.get(ex.get("output_human_language_scope"), ex.get("output_human_language_scope"))
    signal = signal_map.get(raw_label, "non-English or ambiguous audit example")
    key = str(ex.get("instruction_key", ""))[:8]
    ax.text(lefts[0], y_mid, label, fontsize=7.0, va="center")
    ax.text(lefts[1], y_mid, scope, fontsize=6.8, va="center", color=COLORS["slate"])
    ax.text(lefts[2], y_mid, signal, fontsize=6.9, va="center")
    ax.text(lefts[3], y_mid, key, fontsize=6.8, va="center", ha="right", color=COLORS["slate"])
ax.text(0.02, 0.06, "full previews are stored in the language-audit summary; keys retain row-level traceability", fontsize=6.8, color=COLORS["slate"])

save_and_display_figure(fig, "suppfig_s5_2_language_audit_evidence_map")

display(Markdown(
    "**S5-2 language-audit evidence map**  \n"
    f"- Flow cache: `{LANGUAGE_AUDIT_FLOW_CACHE_FILE}`  \n"
    f"- Rows represented: `{flow_summary['rows']:,}`"
))


## Word-Ready Figure Margin Utility

Use this optional cell after generating figures when the manuscript layout needs tighter vertical spacing than the default export. It trims white borders around a PNG and then adds only the requested padding.


In [ ]:
from PIL import Image, ImageChops


def trim_png_for_word(input_file, output_file=None, pad_px=8, bg=(255, 255, 255)):
    """Trim white borders from a PNG and add a controlled uniform pad.

    Use `pad_px=0` for the tightest possible file boundary. Increase only if Word
    places the figure too close to the surrounding text.
    """
    input_path = Path(input_file)
    if not input_path.is_absolute():
        input_path = ROOT / input_path

    if output_file is None:
        output_path = input_path.with_name(f"{input_path.stem}_word_tight{input_path.suffix}")
    else:
        output_path = Path(output_file)
        if not output_path.is_absolute():
            output_path = ROOT / output_path

    image = Image.open(input_path).convert("RGB")
    background = Image.new("RGB", image.size, bg)
    bbox = ImageChops.difference(image, background).getbbox()
    if bbox is None:
        raise ValueError(f"No non-background pixels found in {input_path}")

    left, top, right, bottom = bbox
    left = max(0, left - pad_px)
    top = max(0, top - pad_px)
    right = min(image.width, right + pad_px)
    bottom = min(image.height, bottom + pad_px)

    trimmed = image.crop((left, top, right, bottom))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    trimmed.save(output_path, dpi=image.info.get("dpi", (450, 450)))

    display(trimmed)
    print(f"Original: {image.width} x {image.height}")
    print(f"Trimmed:  {trimmed.width} x {trimmed.height}")
    print(f"Saved to: {output_path}")
    return output_path


# Example: create a Word-tight Figure 1 without touching the canonical export.
trim_png_for_word(
    PQID_ROOT / "submissions" / "scientific_data" / "figures_calibri" / "fig1_pqid_construction_pipeline_designed.png",
    PQID_ROOT / "submissions" / "scientific_data" / "figures_calibri" / "fig1_pqid_construction_pipeline_designed_word_tight.png",
    pad_px=4,
)
